<a href="https://colab.research.google.com/github/Jassmine11/Projects/blob/main/student_mle_miniproject_recommendation_engines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: Recommendation Engines

Recommendation engines are algorithms designed to provide personalized suggestions or recommendations to users. These systems analyze user behavior, preferences, and interactions with items (products, movies, music, articles, etc.) to predict and offer items that users are likely to be interested in. Recommendation engines play a crucial role in enhancing user experience, driving engagement, and increasing conversion rates in various applications, including e-commerce, entertainment, content platforms, and more.

There are generally two approaches taken in collaborative filtering and content-based recommendation engines:

**1. Collaborative Filtering:**
Collaborative Filtering is a popular approach to building recommendation systems that leverages the collective behavior of users to make personalized recommendations. It is based on the idea that users who have agreed in the past will likely agree in the future. There are two main types of collaborative filtering:

- **User-based Collaborative Filtering:** This method finds users similar to the target user based on their past interactions (e.g., ratings or purchases). It then recommends items that similar users have liked but the target user has not interacted with yet.

- **Item-based Collaborative Filtering:** In this approach, the system identifies similar items based on user interactions. It recommends items that are similar to the ones the target user has already liked or interacted with.

Collaborative filtering does not require any explicit information about items but relies on the similarity between users or items. It is effective in capturing complex patterns and can provide serendipitous recommendations. However, it suffers from the cold-start problem (i.e., difficulty in recommending to new users or items with no interactions) and scalability challenges in large datasets.

**2. Content-Based Recommendation:**
Content-based recommendation is an alternative approach to building recommendation systems that focuses on the attributes or features of items and users. It leverages the characteristics of items to make recommendations. The key steps involved in content-based recommendation are:

- **Feature Extraction:** For each item, relevant features are extracted. For movies, these features could be genre, director, actors, and plot summary.

- **User Profile:** A user profile is created based on the items they have interacted with in the past. The user profile contains the weighted importance of features based on their interactions.

- **Similarity Calculation:** The similarity between items or between items and the user profile is calculated using similarity metrics like cosine similarity or Euclidean distance.

- **Recommendation:** Items that are most similar to the user profile are recommended to the user.

Content-based recommendation systems are less affected by the cold-start problem as they can still recommend items based on their features. They are also more interpretable as they rely on item attributes. However, they may miss out on providing serendipitous recommendations and can be limited by the quality of feature extraction and user profiles.

**Choosing Between Collaborative Filtering and Content-Based:**
Both collaborative filtering and content-based approaches have their strengths and weaknesses. The choice between them depends on the specific requirements of the recommendation system, the type of data available, and the user base. Hybrid approaches that combine collaborative filtering and content-based techniques are also common, aiming to leverage the strengths of both methods and mitigate their weaknesses.

In this mini-project, you'll be building both content based and collaborative filtering engines for the [MovieLens 25M dataset](https://grouplens.org/datasets/movielens/25m/). The MovieLens 25M dataset is one of the most widely used and popular datasets for building and evaluating recommendation systems. It is provided by the GroupLens Research project, which collects and studies datasets related to movie ratings and recommendations. The MovieLens 25M dataset contains movie ratings and other related information contributed by users of the MovieLens website.

**Dataset Details:**
- **Size:** The dataset contains approximately 25 million movie ratings.
- **Users:** It includes ratings from over 162,000 users.
- **Movies:** The dataset consists of ratings for more than 62,000 movies.
- **Ratings:** The ratings are provided on a scale of 1 to 5, where 1 is the lowest rating and 5 is the highest.
- **Timestamps:** Each rating is associated with a timestamp, indicating when the rating was given.

**Data Files:**
The dataset is usually split into three CSV files:

1. **movies.csv:** Contains information about movies, including the movie ID, title, genres, and release year.
   - Columns: movieId, title, genres

2. **ratings.csv:** Contains movie ratings provided by users, including the user ID, movie ID, rating, and timestamp.
   - Columns: userId, movieId, rating, timestamp

3. **tags.csv:** Contains user-generated tags for movies, including the user ID, movie ID, tag, and timestamp.
   - Columns: userId, movieId, tag, timestamp

First, import all the libraries you'll need.

In [ ]:
import zipfile
import numpy as np
import pandas as pd
from urllib.request import urlretrieve
from sklearn.metrics.pairwise import cosine_similarity

Next, download the relevant components of the MoveLens dataset. Note, these instructions are roughly based on the colab [here](https://colab.research.google.com/github/google/eng-edu/blob/main/ml/recommendation-systems/recommendation-systems.ipynb?utm_source=ss-recommendation-systems&utm_campaign=colab-external&utm_medium=referral&utm_content=recommendation-systems#scrollTo=O3bcgduFo4s6).

In [ ]:
print("Downloading movielens data...")

urlretrieve('http://files.grouplens.org/datasets/movielens/ml-100k.zip', 'movielens.zip')
zip_ref = zipfile.ZipFile('movielens.zip', 'r')
zip_ref.extractall()
print("Done. Dataset contains:")
print(zip_ref.read('ml-100k/u.info'))

ratings_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t', names=ratings_cols, encoding='latin-1')

# The movies file contains a binary feature for each genre.
genre_cols = [
    "genre_unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]
movies_cols = [
    'movie_id', 'title', 'release_date', "video_release_date", "imdb_url"
] + genre_cols
movies = pd.read_csv(
    'ml-100k/u.item', sep='|', names=movies_cols, encoding='latin-1')

Done. Dataset contains:
b'943 users\n1682 items\n100000 ratings\n'


Before doing any kind of machine learning, it's always good to familiarize yourself with the datasets you'lll be working with.

Here are your tasks:

1. Spend some time familiarizing yourself with both the `movies` and `ratings` dataframes. How many unique user ids are present? How many unique movies are there?
2. Create a new dataframe that merges the `movies` and `ratings` tables on 'movie_id'. Only keep the 'user_id', 'title', 'rating' fields in this new dataframe.

In [ ]:
# Spend some time familiarizing yourself with both the movies and ratings
# dataframes. How many unique user ids are present? How many unique movies
# are there?

# Number of unique users
num_users = ratings['user_id'].nunique()

# Number of unique movies
num_movies = ratings['movie_id'].nunique()

print("Number of unique users:", num_users)
print("Number of unique movies:", num_movies)

Number of unique users: 943
Number of unique movies: 1682


In [ ]:
# Merge movies and ratings dataframes
# Merge on movie_id
merged_df = pd.merge(ratings, movies, on='movie_id')

# Keep only required columns
merged_df = merged_df[['user_id', 'title', 'rating']]

# View result
merged_df.head()

,user_id,title,rating
0,196,Kolya (1996),3
1,186,L.A. Confidential (1997),3
2,22,Heavyweights (1994),1
3,244,Legends of the Fall (1994),2
4,166,Jackie Brown (1997),1


As mentioned in the introduction, content-Based Filtering is a recommendation engine approach that focuses on the attributes or features of items (products, movies, music, articles, etc.) and leverages these features to make personalized recommendations. The underlying idea is to match the characteristics of items with the preferences of users to suggest items that align with their interests. Content-based filtering is particularly useful when explicit user-item interactions (e.g., ratings or purchases) are sparse or unavailable.

**Key Steps in Content-Based Filtering:**

1. **Feature Extraction:**
   - For each item, relevant features are extracted. These features are typically descriptive attributes that can be represented numerically, such as genre, director, actors, author, publication date, and keywords.
   - In the case of text-based items, natural language processing techniques may be used to extract features like TF-IDF (Term Frequency-Inverse Document Frequency) scores.

2. **User Profile Creation:**
   - A user profile is created based on the items they have interacted with in the past. The user profile contains the weighted importance of features based on their interactions.
   - For example, if a user has watched several action movies, the action genre feature would receive a higher weight in their profile.

3. **Similarity Calculation:**
   - The similarity between items or between items and the user profile is calculated using similarity metrics like cosine similarity, Euclidean distance, or Pearson correlation.
   - Cosine similarity is commonly used as it measures the cosine of the angle between two vectors, which represents their similarity.

4. **Recommendation:**
   - Items that are most similar to the user profile are recommended to the user. These are items whose features have the highest similarity scores with the user profile.
   - The recommended items are presented as a list sorted by their similarity scores.

**Advantages of Content-Based Filtering:**
1. **No Cold-Start Problem:** Content-based filtering can make recommendations even for new users with no historical interactions because it relies on item features rather than user history.

2. **User Independence:** The recommendations are based solely on the features of items and do not require knowledge of other users' preferences or behavior.

3. **Transparency:** Content-based recommendations are interpretable, as they depend on the features of items, making it easier for users to understand why specific items are recommended.

4. **Serendipity:** Content-based filtering can recommend items with characteristics not seen before by the user, leading to serendipitous discoveries.

5. **Diversity in Recommendations:** The method can offer diverse recommendations since it suggests items with different feature combinations.

**Limitations of Content-Based Filtering:**
1. **Limited Discovery:** Content-based filtering may struggle to recommend items outside the scope of users' historical interactions or interests.

2. **Over-Specialization:** Users may receive recommendations that are too similar to their previous choices, leading to a lack of exposure to new item categories.

3. **Dependency on Feature Quality:** The quality and relevance of item features significantly influence the quality of recommendations.

4. **Limited for Cold Items:** Content-based filtering can struggle to recommend new items with limited feature information.

Here is your task:

1. Write a function that takes in a user id and the dataframe you created before that contains 'user_id', 'title', and 'rating'. The function should return content-based recommendations for this user. Here are steps you can take:

  A. Get the user's rated movies

  B. Create a TF-IDF matrix using movie genres. Note, this can be extracted from the `movies` dataframe.

  C. Compute the cosine similarity between movie genres. Use the [cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) function.

  D. Get the indices of similar movies to those rated by the user based on cosine similarity. Keep only the top 5.

  E. Remove duplicates and movies already rated by the user.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def content_based_recommendation(user_id, df):

    # A. Get user's rated movies
    user_movies = df[df['user_id'] == user_id]['title'].unique()

    # B. Create TF-IDF matrix using movie genres
    genre_cols = [
        "genre_unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
        "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
        "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
    ]

    movie_data = movies[['title'] + genre_cols].copy()

    # Convert binary genre columns into text format
    movie_data['genres'] = movie_data[genre_cols].apply(
        lambda x: ' '.join(x.index[x == 1]),
        axis=1
    )

    movie_data = movie_data[['title', 'genres']].reset_index(drop=True)

    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(movie_data['genres'])

    # C. Compute cosine similarity
    cosine_sim = cosine_similarity(tfidf_matrix)

    # Map titles to indices
    indices = pd.Series(movie_data.index, index=movie_data['title'])

    recommendations = []

    # D. Get similar movies for each rated movie
    for movie in user_movies:
        if movie in indices:

            idx = indices[movie]

            sim_scores = list(enumerate(cosine_sim[idx].ravel()))
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

            top_5 = []

            for i in sim_scores[1:6]:
                if i[0] < len(movie_data):
                    top_5.append(movie_data.iloc[i[0]]['title'])

            recommendations.extend(top_5)

    # E. Remove duplicates and already rated movies
    recommendations = list(dict.fromkeys(recommendations))
    recommendations = [m for m in recommendations if m not in user_movies]

    return recommendations[:5]

In [ ]:
print(content_based_recommendation(196, merged_df))

['Brothers McMullen, The (1995)', 'To Wong Foo, Thanks for Everything! Julie Newmar (1995)', 'Billy Madison (1995)', 'Clerks (1994)', 'I.Q. (1994)']


The key idea behind collaborative filtering is that users who have agreed in the past will likely agree in the future. Instead of relying on item attributes or user profiles, collaborative filtering identifies patterns of user behavior and item preferences from the interactions present in the data.

**Types of Collaborative Filtering:**
There are two main types of collaborative filtering:

**Collaborative Filtering Process:**
The collaborative filtering process typically involves the following steps:

1. **Data Collection:**
   - Gather data on user-item interactions, such as movie ratings, product purchases, or article clicks.

2. **User-Item Matrix:**
   - Organize the data into a user-item matrix, where rows represent users, columns represent items, and the entries contain the users' interactions (e.g., ratings).

3. **Similarity Calculation:**
   - Calculate the similarity between users or items using similarity metrics such as cosine similarity, Pearson correlation, or Jaccard similarity.
   - For user-based collaborative filtering, user similarities are calculated, and for item-based collaborative filtering, item similarities are calculated.

4. **Neighborhood Selection:**
   - For each user or item, select the most similar users or items as the neighborhood.
   - The size of the neighborhood (the number of similar users or items to consider) is an important parameter to control the system's behavior.

5. **Prediction Generation:**
   - Predict the ratings for items that the target user has not yet interacted with by combining the ratings of neighboring users or items.

6. **Recommendation Generation:**
   - Recommend items with the highest predicted ratings to the target user.

**Advantages of Collaborative Filtering using User-Item Interactions:**
- Collaborative filtering is based solely on user interactions and does not require knowledge of item attributes, making it useful for cases where item data is sparse or unavailable.
- It can provide serendipitous recommendations, suggesting items that users may not have discovered on their own.
- Collaborative filtering can be applied in various domains, including e-commerce, music, movie, and content recommendations.

**Limitations of Collaborative Filtering:**
- The cold-start problem: Collaborative filtering struggles to recommend to new users or items with no or limited interaction history.
- It may suffer from sparsity when data is limited or when users have only interacted with a small subset of items.
- Scalability issues can arise with large datasets and an increasing number of users or items.

Here is your task:

1. Write a function that takes in a user id and the dataframe you created before that contains 'user_id', 'title', and 'rating'. The function should return collaborative filtering recommendations for this user based on a user-item interaction matrix. Here are steps you can take:

  A. Create the user-item matrix using Pandas' [pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html).

  B. Fill missing values with zeros in this matrix.

  C. Calculate user-user similarity matrix using cosine similarity.

  D. Get the array of similarity scores of the target user with all other users from the similarity matrix.

  E. Extract, say the the top 5 most similar users (excluding the target user).

  F. Generate movie recommendations based on the most similar users.

  G. Remove duplicate movies recommendations.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def collaborative_filtering_recommendation(user_id, df):

    # A. Create user-item matrix
    user_item = df.pivot_table(
        index='user_id',
        columns='title',
        values='rating'
    ).fillna(0)

    # B. Ensure user exists
    if user_id not in user_item.index:
        return []

    # C. User-user similarity
    user_similarity = cosine_similarity(user_item)


    sim_df = pd.DataFrame(
        user_similarity,
        index=user_item.index,
        columns=user_item.index
    )

    # D. Get similarity scores for target user
    sim_scores = sim_df[user_id].sort_values(ascending=False)

    # E. Top 5 similar users (excluding self)
    similar_users = sim_scores.iloc[1:6].index

    # F. Get movies watched by similar users
    similar_users_ratings = user_item.loc[similar_users]

    # Average ratings from similar users
    movie_scores = similar_users_ratings.mean().sort_values(ascending=False)

    # Movies already seen by target user
    seen_movies = set(df[df['user_id'] == user_id]['title'])

    recommendations = [movie for movie in movie_scores.index if movie not in seen_movies]

    # G. Remove duplicates (safe)
    recommendations = list(dict.fromkeys(recommendations))

    return recommendations[:5]

In [ ]:
print(collaborative_filtering_recommendation(196, merged_df))

['Back to the Future (1985)', 'Indiana Jones and the Last Crusade (1989)', 'Fargo (1996)', 'Monty Python and the Holy Grail (1974)', 'Sting, The (1973)']


Now, test your recommendations engines! Select a few user ids and generate recommendations using both functions you've written. Are the recommendations similar? Do the recommendations make sense?

In [ ]:
users = [196, 186, 22]

for u in users:
    print("\n============================")
    print("User ID:", u)

    print("\n🎯 Content-Based Recommendations:")
    print(content_based_recommendation(u, merged_df))

    print("\n🤝 Collaborative Filtering Recommendations:")
    print(collaborative_filtering_recommendation(u, merged_df))


User ID: 196

🎯 Content-Based Recommendations:
['Brothers McMullen, The (1995)', 'To Wong Foo, Thanks for Everything! Julie Newmar (1995)', 'Billy Madison (1995)', 'Clerks (1994)', 'I.Q. (1994)']

🤝 Collaborative Filtering Recommendations:
['Back to the Future (1985)', 'Indiana Jones and the Last Crusade (1989)', 'Fargo (1996)', 'Monty Python and the Holy Grail (1974)', 'Sting, The (1973)']

User ID: 186

🎯 Content-Based Recommendations:
['Laura (1944)', 'Chinatown (1974)', 'Palmetto (1998)', 'Touch of Evil (1958)', 'Daylight (1996)']

🤝 Collaborative Filtering Recommendations:
['Jurassic Park (1993)', 'Godfather, The (1972)', 'Terminator 2: Judgment Day (1991)', 'Seven (Se7en) (1995)', 'Return of the Jedi (1983)']

User ID: 22

🎯 Content-Based Recommendations:
['D3: The Mighty Ducks (1996)', 'Love Bug, The (1969)', '101 Dalmatians (1996)', 'Jungle2Jungle (1997)', 'Birdcage, The (1996)']

🤝 Collaborative Filtering Recommendations:
['Alien (1979)', 'Braveheart (1995)', 'Jurassic Park (

In this project, two recommendation systems were developed and evaluated using the MovieLens dataset: a content-based filtering model and a collaborative filtering model.

Content-Based Recommendation System

The content-based approach recommends movies based on the similarity of item features. In this project, movie genres were used as the primary feature representation. Each movie was converted into a text-based feature vector using TF-IDF, and cosine similarity was computed between all movies.

For a given user, the system first identifies movies they have previously rated. It then finds movies with the highest similarity in genre space and recommends the top similar items, excluding movies already seen by the user. This method ensures that recommendations closely match the user’s past preferences in terms of movie content and style.

Collaborative Filtering Recommendation System

The collaborative filtering approach is based on user behavior rather than item features. A user-item interaction matrix was constructed using user ratings. Cosine similarity was then applied to measure similarity between users.

For a target user, the system identifies the top similar users and recommends movies that those users have highly rated but the target user has not yet seen. This method captures hidden patterns in user behavior and allows for more diverse and sometimes unexpected recommendations.

Results and Observations

Both models were tested on multiple users, and their outputs showed clear differences:

The content-based model produced recommendations that were very similar in genre and style to previously rated movies.
The collaborative filtering model produced more diverse recommendations, often including popular and highly rated movies across different genres.
For example, users who preferred comedies received similar comedy-based recommendations from the content-based model, while collaborative filtering suggested broader classic and popular films.
Conclusion

This project demonstrates the fundamental differences between content-based and collaborative filtering approaches. Content-based filtering focuses on item similarity, while collaborative filtering leverages collective user behavior. Both methods have strengths and limitations, and combining them in a hybrid system could further improve recommendation quality.